# Part 3: Frequent itemsets and association rules with Apriori and PCY

In this notebook we find the products (and aisles and departments) that are frequently bought together, and turn them into association rules such as "customers who buy X also tend to buy Y". These rules are the basis for combo offers and for the product, aisle and department queries in the recommendation step.

The core algorithms (Apriori, PCY and SON) are written from scratch with numpy. A library (mlxtend's FP-Growth) is only used at the end of the correctness check, to confirm our answers are right.

We only use the "prior" orders here. The "train" orders are each user's held-out last order, used later for evaluation, so mining rules from them would leak the answers into the model.

In [ ]:
# Load the baskets and lookup tables from preprocessing, keeping prior orders only.

# Requires pip install pyarrow numpy pandas scipy matplotlib (and mlxtend, only for the correctness check)

import os
import time
import itertools
import tracemalloc
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.compute as pc
import pyarrow.parquet as pq
from scipy import sparse

save_path = "processed/"
results_path = "results/"     # Experiment result tables are saved here as .csv files.
SEED = 612

print ("~~~~ Loading... ~~~~")

baskets_table = pq.read_table(os.path.join(save_path, "baskets.parquet"), columns = ["order_id", "product_id"])
products_lookup = pd.read_parquet(os.path.join(save_path, "products_lookup.parquet"), engine = "pyarrow")
test_order_ids = pd.read_parquet(os.path.join(save_path, "user_test_orders.parquet"), engine = "pyarrow", columns = ["order_id"])["order_id"].to_numpy()

print ("\t baskets has shape:\t\t", baskets_table.shape)
print ("\t products_lookup has shape:\t", products_lookup.shape)
print ("\t held-out test orders:\t\t", len(test_order_ids))
print ("~~~~ Load complete. ~~~~")

# Keep prior orders only (drop each user's held-out "train" order to avoid a train/test leak).
is_prior = ~np.isin(baskets_table["order_id"].to_numpy(), test_order_ids)
product_lists = baskets_table.filter(is_prior)["product_id"].combine_chunks()

# Store all baskets as one flat array of product IDs plus an offset per basket (basket b is flat_items[offsets[b]:offsets[b + 1]]).
flat_items = product_lists.flatten().to_numpy().astype(np.int32)
offsets = np.r_[0, np.cumsum(pc.list_value_length(product_lists).to_numpy())].astype(np.int64)
n_baskets = len(offsets) - 1
n_products = int(flat_items.max()) + 1     # Product IDs are below this, so it is also the base for encoding itemsets.
del baskets_table, product_lists

# Every algorithm below assumes a product appears at most once per basket, so we check that here.
basket_of_item = np.repeat(np.arange(n_baskets, dtype = np.int64), np.diff(offsets))
repeats = len(flat_items) - len(np.unique(basket_of_item * n_products + flat_items))
del basket_of_item

# Product names, indexed by product ID, for displaying results.
product_names = np.full(n_products, "", dtype = object)
product_names[products_lookup["product_id"].to_numpy()] = products_lookup["product_name"].to_numpy()

print ("\nPrior baskets:\t\t\t", n_baskets)
print ("Items in all baskets:\t\t", len(flat_items))
print ("Repeated items within a basket:\t", repeats)
print ("Basket sizes:\n", pd.Series(np.diff(offsets)).describe())

## Pass 1: Counting items

The first pass counts how many baskets contain each item. Since every item appears at most once per basket, this is a single np.bincount over the flattened product IDs.

An item is frequent if it appears in at least the support threshold s of baskets. We start at 0.1% of baskets (about 3.2K orders), and try other thresholds in the experiments at the end.

The frequent items are then renumbered 0..m-1. This lets us store the count of every pair of frequent items in a triangular array (section 6.2.2) with no wasted space, instead of an m x m matrix where half the cells are never used.

Since all later passes look at every basket in the same way, they share a few helper functions: baskets are processed in chunks, and within a chunk all baskets of the same size are handled together, so the pairs (or triples) inside them are generated in one vectorized step instead of a Python loop over 3.2M baskets.

In [ ]:
# Helper functions shared by every algorithm, and Pass 1: count every item and keep the frequent ones.

SUPPORT_FRACTION = 0.001      # Start at 0.1% of baskets.
CHUNK_BASKETS = 100_000       # Baskets processed per chunk, to keep memory use small.

def support_threshold(fraction, n_baskets):
  # Minimum number of baskets an itemset must appear in. The small epsilon avoids float rounding errors.
  return int(np.ceil(fraction * n_baskets - 1e-9))

def timed(function, *args):
  # Run a function and return its result, time in seconds, and peak memory in MB (tracked with tracemalloc).
  tracemalloc.start()
  start_time = time.perf_counter()
  result = function(*args)
  elapsed = time.perf_counter() - start_time
  _, peak = tracemalloc.get_traced_memory()
  tracemalloc.stop()
  return result, elapsed, peak / 1e6

def basket_chunk(flat_items, offsets, item_map, start, stop):
  # Items of baskets start..stop-1 after renumbering with item_map (-1 drops an item), sorted and
  # de-duplicated within each basket. Returns the items, the start position of each basket and each basket's size.
  items = item_map[flat_items[offsets[start]:offsets[stop]]]
  basket = np.repeat(np.arange(stop - start), np.diff(offsets[start:stop + 1]))
  keep = items >= 0
  items, basket = items[keep], basket[keep]

  if len(items):
    order = np.lexsort((items, basket))
    items, basket = items[order], basket[order]
    first = np.r_[True, (items[1:] != items[:-1]) | (basket[1:] != basket[:-1])]
    items, basket = items[first], basket[first]

  sizes = np.bincount(basket, minlength = stop - start)
  starts = np.r_[0, np.cumsum(sizes)[:-1]].astype(np.int64)
  return items, starts, sizes

@lru_cache(maxsize = None)
def combination_index(size, k):
  # Positions of every k-item combination in a basket of this size, e.g. (0, 1), (0, 2), (1, 2) for size 3 and k = 2.
  return np.array(list(itertools.combinations(range(size), k)), dtype = np.int64)

def basket_combinations(items, starts, sizes, k, max_rows = 2_000_000):
  # Yields every k-item combination inside each basket, as (combinations x k) arrays of at most about max_rows rows.
  # Baskets of the same size are handled together, so each batch is made in one vectorized step.
  for size in np.unique(sizes):
    if size < k:
      continue
    index = combination_index(int(size), k)
    group_starts = starts[sizes == size]
    batch = max(1, max_rows // len(index))
    for first in range(0, len(group_starts), batch):
      positions = group_starts[first:first + batch, None, None] + index[None, :, :]
      yield items[positions].reshape(-1, k)

def merge_counts(keys, counts):
  # Add up the counts of equal keys, given lists of (keys, counts) arrays from several chunks.
  keys = np.concatenate(keys)
  counts = np.concatenate(counts)
  unique_keys, inverse = np.unique(keys, return_inverse = True)
  return unique_keys, np.bincount(inverse.ravel(), weights = counts, minlength = len(unique_keys)).astype(np.int64)

def decode_keys(keys, n_items, k):
  # Itemsets are stored as one int64 key (e.g. a * n_items + b for a pair). Turn keys back into a (n x k) array of item IDs.
  columns = []
  for _ in range(k):
    columns.append(keys % n_items)
    keys = keys // n_items
  return np.stack(columns[::-1], axis = 1)

def count_items(flat_items, n_items, support):
  # Pass 1: count every item, keep the frequent ones and renumber them 0..m-1 (item_map is -1 for infrequent items).
  item_counts = np.bincount(flat_items, minlength = n_items)
  frequent_items = np.flatnonzero(item_counts >= support).astype(np.int32)
  item_map = np.full(n_items, -1, dtype = np.int32)
  item_map[frequent_items] = np.arange(len(frequent_items), dtype = np.int32)
  return item_counts, frequent_items, item_map

support = support_threshold(SUPPORT_FRACTION, n_baskets)
(item_counts, frequent_items, item_map), apriori_pass1_s, apriori_pass1_MB = timed(count_items, flat_items, n_products, support)
m = len(frequent_items)

print ("~~~~ Pass 1 complete ~~~~")
print ("Support threshold:\t", support, "baskets (", SUPPORT_FRACTION * 100, "% )")
print ("Frequent items (m):\t", m, "of", int((item_counts > 0).sum()))
print ("Time:\t\t\t", round(apriori_pass1_s, 2), "seconds")

top_items = np.argsort(-item_counts)[:10]
display (pd.DataFrame({"product_id": top_items, "product_name": product_names[top_items],
                       "baskets": item_counts[top_items], "support": item_counts[top_items] / n_baskets}))

## Apriori pass 2: Counting pairs

By monotonicity, a pair can only be frequent if both of its items are frequent. So in the second pass we drop the infrequent items from each basket and count only the pairs of frequent items.

The counts go into a triangular array of m(m-1)/2 int32 values. With items renumbered 0..m-1, pair (i, j) with i < j is stored at index i(2m - i - 1)/2 + (j - i - 1), the 0-based version of the formula in section 6.2.2. For each chunk of baskets, we compute the index of every pair and add them all to the array with one np.bincount.

Frequent itemsets are stored as a single int64 key, e.g. a * n_products + b for the pair (a, b) of original product IDs, so they are easy to sort, compare and save.

In [ ]:
# Apriori pass 2: count every pair of frequent items in a triangular array.

def apriori_pairs(flat_items, offsets, frequent_items, item_map, support, n_items, chunk_baskets = CHUNK_BASKETS):
  # Returns the frequent pairs (keys a * n_items + b, sorted), their counts, and stats about the counting structure.
  m = len(frequent_items)
  n_baskets = len(offsets) - 1
  rows = np.arange(m, dtype = np.int64)
  row_start = rows * (2 * m - rows - 1) // 2            # Index of pair (i, i + 1) in the triangular array.
  pair_counts = np.zeros(m * (m - 1) // 2, dtype = np.int32)

  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, starts, sizes = basket_chunk(flat_items, offsets, item_map, start, stop)
    index = [row_start[pairs[:, 0]] + (pairs[:, 1] - pairs[:, 0] - 1) for pairs in basket_combinations(items, starts, sizes, 2)]
    if index:
      pair_counts += np.bincount(np.concatenate(index), minlength = len(pair_counts))

  # Turn the triangular indexes of frequent pairs back into (i, j), then into original item IDs.
  frequent_index = np.flatnonzero(pair_counts >= support)
  i = np.searchsorted(row_start, frequent_index, side = "right") - 1
  j = frequent_index - row_start[i] + i + 1
  keys = frequent_items[i].astype(np.int64) * n_items + frequent_items[j]

  stats = {"candidate_pairs": len(pair_counts), "structure_MB": pair_counts.nbytes / 1e6}
  return keys, pair_counts[frequent_index].astype(np.int64), stats

(pair_keys, pair_counts, apriori_stats), apriori_pass2_s, apriori_pass2_MB = timed(
    apriori_pairs, flat_items, offsets, frequent_items, item_map, support, n_products)

print ("~~~~ Apriori pass 2 complete ~~~~")
print ("Candidate pairs (m(m-1)/2):\t", apriori_stats["candidate_pairs"])
print ("Triangular array size:\t\t", round(apriori_stats["structure_MB"], 1), "MB")
print ("Frequent pairs:\t\t\t", len(pair_keys))
print ("Time:\t\t\t\t", round(apriori_pass2_s, 2), "seconds")

top_pairs = np.argsort(-pair_counts)[:10]
pairs = decode_keys(pair_keys[top_pairs], n_products, 2)
display (pd.DataFrame({"item_a": product_names[pairs[:, 0]], "item_b": product_names[pairs[:, 1]],
                       "baskets": pair_counts[top_pairs], "support": pair_counts[top_pairs] / n_baskets}))

## Triples

Three-item bundles make good combo offers, so we run Apriori one more level, to k = 3, if the threshold leaves any frequent triples.

Again by monotonicity, a triple can only be frequent if all three of its pairs are frequent. Only items that are part of some frequent pair can be in a frequent triple, so each basket is first reduced to those items. We then generate the triples in each basket, keep the candidates whose three pairs are all frequent (looked up in a small boolean matrix), and count them.

The candidate triples are scattered, so instead of an array with a slot for every possible triple, we count only the ones that occur (np.unique per chunk, then the chunk counts are merged).

In [ ]:
# Apriori pass 3: count candidate triples, whose three pairs are all frequent.

def apriori_triples(flat_items, offsets, pair_keys, support, n_items, chunk_baskets = CHUNK_BASKETS):
  # Returns the frequent triples (keys (a * n_items + b) * n_items + c, sorted), their counts, and stats.
  empty = np.empty(0, dtype = np.int64)
  pairs = decode_keys(pair_keys, n_items, 2)
  pair_items = np.unique(pairs).astype(np.int32)      # Only items in a frequent pair can be in a frequent triple.
  q = len(pair_items)
  if q < 3:
    return empty, empty, {"candidate_triples": 0}

  item_map = np.full(n_items, -1, dtype = np.int32)
  item_map[pair_items] = np.arange(q, dtype = np.int32)
  frequent_pair = np.zeros((q, q), dtype = bool)
  frequent_pair[item_map[pairs[:, 0]], item_map[pairs[:, 1]]] = True

  n_baskets = len(offsets) - 1
  keys, counts = [], []
  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, starts, sizes = basket_chunk(flat_items, offsets, item_map, start, stop)

    chunk_keys = []
    for triples in basket_combinations(items, starts, sizes, 3):
      a, b, c = triples[:, 0], triples[:, 1], triples[:, 2]
      candidate = frequent_pair[a, b] & frequent_pair[a, c] & frequent_pair[b, c]
      chunk_keys.append((a[candidate].astype(np.int64) * q + b[candidate]) * q + c[candidate])

    if chunk_keys:
      chunk_keys, chunk_counts = np.unique(np.concatenate(chunk_keys), return_counts = True)
      keys.append(chunk_keys)
      counts.append(chunk_counts)

    # Merge every 10 chunks, so the per-chunk counts do not pile up in RAM.
    if len(keys) >= 10:
      merged = merge_counts(keys, counts)
      keys, counts = [merged[0]], [merged[1]]

  if not keys:
    return empty, empty, {"candidate_triples": 0}
  keys, counts = merge_counts(keys, counts)
  frequent = counts >= support

  # Back to original item IDs. pair_items is sorted, so the keys stay sorted.
  triples = pair_items[decode_keys(keys[frequent], q, 3)].astype(np.int64)
  triple_keys = (triples[:, 0] * n_items + triples[:, 1]) * n_items + triples[:, 2]
  return triple_keys, counts[frequent], {"candidate_triples": len(keys)}

(triple_keys, triple_counts, triple_stats), apriori_pass3_s, apriori_pass3_MB = timed(
    apriori_triples, flat_items, offsets, pair_keys, support, n_products)

print ("~~~~ Apriori pass 3 complete ~~~~")
print ("Candidate triples that occur:\t", triple_stats["candidate_triples"])
print ("Frequent triples:\t\t", len(triple_keys))
print ("Time:\t\t\t\t", round(apriori_pass3_s, 2), "seconds")

top_triples = np.argsort(-triple_counts)[:10]
triples = decode_keys(triple_keys[top_triples], n_products, 3)
display (pd.DataFrame({"item_a": product_names[triples[:, 0]], "item_b": product_names[triples[:, 1]],
                       "item_c": product_names[triples[:, 2]], "baskets": triple_counts[top_triples]}))

## Checking that the answers are right

We check the results in two independent ways:

1. Pairs on the full data: we build a sparse basket x item matrix X of the frequent items. Entry (i, j) of X^T X is then the number of baskets containing both i and j, computed by scipy with no code shared with our Apriori. The frequent pairs from both methods must match exactly.
2. Itemsets up to size 3 on a sample of baskets: we run mlxtend's FP-Growth (a completely different algorithm) and our Apriori on the same sample with the same threshold, and compare the frequent itemsets. FP-Growth needs a one-hot table of the baskets, so we only use it on a sample, and only as a check.

In [ ]:
# Check the frequent pairs against a sparse matrix product, and the itemsets against mlxtend's FP-Growth on a sample.

CHECK_BASKETS = 50_000
CHECK_SUPPORT_FRACTION = 0.005
rng_check = np.random.default_rng(SEED)

def run_apriori(flat_items, offsets, n_items, support, max_k = 3):
  # Full Apriori up to max_k (2 or 3). Returns {k: (sorted keys, counts)} and stats about each pass.
  item_counts, frequent_items, item_map = count_items(flat_items, n_items, support)
  itemsets = {1: (frequent_items.astype(np.int64), item_counts[frequent_items].astype(np.int64))}
  pair_keys, pair_counts, stats = apriori_pairs(flat_items, offsets, frequent_items, item_map, support, n_items)
  itemsets[2] = (pair_keys, pair_counts)
  stats["frequent_items"] = len(frequent_items)
  if max_k >= 3:
    triple_keys, triple_counts, triple_stats = apriori_triples(flat_items, offsets, pair_keys, support, n_items)
    itemsets[3] = (triple_keys, triple_counts)
    stats.update(triple_stats)
  return itemsets, stats

def subset_baskets(flat_items, offsets, rows):
  # Flat items and offsets for a subset of baskets (rows sorted), in the same format as the full data.
  sizes = offsets[rows + 1] - offsets[rows]
  new_offsets = np.r_[0, np.cumsum(sizes)].astype(np.int64)
  positions = np.repeat(offsets[rows] - new_offsets[:-1], sizes) + np.arange(new_offsets[-1])
  return flat_items[positions], new_offsets

# Check 1: pair counts from a sparse matrix product X^T X on the full data.
kept = item_map[flat_items] >= 0
X = sparse.csr_matrix((np.ones(int(kept.sum()), dtype = np.int32), item_map[flat_items[kept]],
                       np.r_[0, np.cumsum(np.add.reduceat(kept.astype(np.int64), offsets[:-1]))]), shape = (n_baskets, m))
co_occurrence = sparse.triu(X.T @ X, k = 1).tocoo()
frequent = co_occurrence.data >= support
check_keys = frequent_items[co_occurrence.row[frequent]].astype(np.int64) * n_products + frequent_items[co_occurrence.col[frequent]]
check_order = np.argsort(check_keys)
pairs_match = np.array_equal(check_keys[check_order], pair_keys) and np.array_equal(co_occurrence.data[frequent][check_order], pair_counts)
del X, co_occurrence, kept

print ("~~~~ Check 1: sparse matrix product (full data) ~~~~")
print ("Frequent pairs from X^T X:\t", len(check_keys))
print ("Same pairs and counts as Apriori:\t", pairs_match)

# Check 2: mlxtend FP-Growth on a sample of baskets.
sample_rows = np.sort(rng_check.choice(n_baskets, size = min(CHECK_BASKETS, n_baskets), replace = False))
sample_items, sample_offsets = subset_baskets(flat_items, offsets, sample_rows)
sample_support = support_threshold(CHECK_SUPPORT_FRACTION, len(sample_rows))
our_itemsets, _ = run_apriori(sample_items, sample_offsets, n_products, sample_support)
our_sets = {frozenset(row.tolist()): int(count) for k in (1, 2, 3)
            for row, count in zip(decode_keys(our_itemsets[k][0], n_products, k), our_itemsets[k][1])}

try:
  from mlxtend.frequent_patterns import fpgrowth

  # One-hot table of the sample: one boolean column per product that appears in it.
  sample_products, columns = np.unique(sample_items, return_inverse = True)
  one_hot = sparse.csr_matrix((np.ones(len(sample_items), dtype = bool), columns.ravel(), sample_offsets),
                              shape = (len(sample_rows), len(sample_products)))
  # mlxtend needs string column names for sparse tables, so product IDs are converted to strings and back.
  one_hot = pd.DataFrame.sparse.from_spmatrix(one_hot, columns = [str(i) for i in sample_products])
  library_itemsets = fpgrowth(one_hot, min_support = CHECK_SUPPORT_FRACTION, use_colnames = True, max_len = 3)
  library_sets = {frozenset(int(i) for i in items): int(round(s * len(sample_rows)))
                  for items, s in zip(library_itemsets["itemsets"], library_itemsets["support"])}

  print ("\n~~~~ Check 2: mlxtend FP-Growth (sample of", len(sample_rows), "baskets, support", sample_support, ") ~~~~")
  for k in (1, 2, 3):
    print ("Frequent itemsets of size", k, "- ours:", sum(len(s) == k for s in our_sets), "\tFP-Growth:", sum(len(s) == k for s in library_sets))
  print ("Same itemsets and counts as FP-Growth:\t", our_sets == library_sets)
  del one_hot, library_itemsets
except ImportError:
  print ("\nmlxtend is not installed (pip install mlxtend), so the FP-Growth check was skipped.")

## PCY

The PCY algorithm (Park, Chen and Yu) uses the memory that sits idle during Apriori's first pass. While counting items in pass 1, it also hashes every pair in each basket into a large array of bucket counts. A bucket is frequent if its count reaches the support threshold. Since a frequent pair always lands in a frequent bucket, any pair in an infrequent bucket cannot be frequent.

After pass 1, the bucket counts are replaced by a bitmap with 1 bit per bucket (1 = frequent), which takes 1/32 of the space of the int32 counts. In pass 2, a pair is counted only if both items are frequent and its bucket is frequent.

The surviving candidate pairs are scattered, so PCY cannot use the triangular array. Instead it counts them as (i, j, count) triples, which take 12 bytes per candidate pair instead of 4. PCY only saves memory in pass 2 if it removes at least 2/3 of the candidate pairs.

We use 16,777,213 buckets (a prime just under 2^24) and the hash function h(i, j) = (i * 1,000,003 + j) mod the number of buckets.

In [ ]:
# PCY: pass 1 also hashes every pair into buckets, and pass 2 counts a pair only if its bucket is frequent.

N_BUCKETS = 16_777_213          # A large prime number of buckets, just under 2^24.
HASH_MULTIPLIER = 1_000_003

def pair_bucket(a, b, n_buckets = N_BUCKETS):
  # Hash a pair of original item IDs (a < b) into one of n_buckets buckets.
  return (a.astype(np.int64) * HASH_MULTIPLIER + b) % n_buckets

def in_frequent_bucket(bitmap, buckets):
  # Look up each bucket's bit in the bitmap (np.packbits stores the first bucket in the highest bit of each byte).
  return ((bitmap[buckets >> 3] >> (7 - (buckets & 7))) & 1).astype(bool)

def pcy_pass1(flat_items, offsets, n_items, support, n_buckets = N_BUCKETS, chunk_baskets = CHUNK_BASKETS):
  # Pass 1 of PCY: count items like Apriori, and also hash every pair in each basket into a bucket array.
  item_counts, frequent_items, item_map = count_items(flat_items, n_items, support)
  bucket_counts = np.zeros(n_buckets, dtype = np.int32)
  all_items = np.arange(n_items, dtype = np.int32)      # Frequent items are not known yet in pass 1, so every pair is hashed.

  n_baskets = len(offsets) - 1
  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, starts, sizes = basket_chunk(flat_items, offsets, all_items, start, stop)
    buckets = [pair_bucket(pairs[:, 0], pairs[:, 1], n_buckets) for pairs in basket_combinations(items, starts, sizes, 2)]
    if buckets:
      bucket_counts += np.bincount(np.concatenate(buckets), minlength = n_buckets)

  # Replace the bucket counts with a bitmap of frequent buckets: 1 bit per bucket instead of 4 bytes.
  frequent_buckets = bucket_counts >= support
  bitmap = np.packbits(frequent_buckets)
  stats = {"frequent_bucket_share": float(frequent_buckets.mean()),
           "bucket_array_MB": bucket_counts.nbytes / 1e6, "bitmap_MB": bitmap.nbytes / 1e6}
  return item_counts, frequent_items, item_map, bitmap, stats

def pcy_pass2(flat_items, offsets, frequent_items, item_map, bitmap, support, n_items, n_buckets = N_BUCKETS, chunk_baskets = CHUNK_BASKETS):
  # Pass 2 of PCY: count a pair only if both items are frequent AND its bucket is frequent.
  # Returns the frequent pairs (keys a * n_items + b, sorted), their counts, and stats.
  m = len(frequent_items)
  n_baskets = len(offsets) - 1
  keys, counts = [], []

  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, starts, sizes = basket_chunk(flat_items, offsets, item_map, start, stop)

    chunk_keys = []
    for pairs in basket_combinations(items, starts, sizes, 2):
      candidate = in_frequent_bucket(bitmap, pair_bucket(frequent_items[pairs[:, 0]], frequent_items[pairs[:, 1]], n_buckets))
      chunk_keys.append(pairs[candidate, 0].astype(np.int64) * m + pairs[candidate, 1])

    if chunk_keys:
      chunk_keys, chunk_counts = np.unique(np.concatenate(chunk_keys), return_counts = True)
      keys.append(chunk_keys)
      counts.append(chunk_counts)

    # Merge every 10 chunks, so the per-chunk counts do not pile up in RAM.
    if len(keys) >= 10:
      merged = merge_counts(keys, counts)
      keys, counts = [merged[0]], [merged[1]]

  keys, counts = merge_counts(keys, counts) if keys else (np.empty(0, dtype = np.int64), np.empty(0, dtype = np.int64))
  frequent = counts >= support
  pairs = decode_keys(keys[frequent], m, 2)
  pair_keys = frequent_items[pairs[:, 0]].astype(np.int64) * n_items + frequent_items[pairs[:, 1]]
  return pair_keys, counts[frequent], {"counted_pairs": len(keys)}

def count_pcy_candidates(frequent_items, bitmap, n_buckets = N_BUCKETS):
  # Number of PCY candidate pairs: pairs of frequent items whose bucket is frequent (checked over all m(m-1)/2 pairs).
  total = 0
  for i in range(len(frequent_items) - 1):
    a = np.full(len(frequent_items) - i - 1, frequent_items[i])
    total += int(in_frequent_bucket(bitmap, pair_bucket(a, frequent_items[i + 1:], n_buckets)).sum())
  return total

We now run both passes of PCY and compare them with Apriori's passes from above, on the same data and threshold. For each method we report the number of candidate pairs, the memory of the counting structure, the measured peak memory and the runtime of each pass.

For the counting structure, Apriori uses the triangular array (4 bytes per candidate pair), and PCY uses the bitmap plus 12 bytes per candidate pair, as in the textbook. Both methods must return exactly the same frequent pairs.

In [ ]:
# Compare PCY with Apriori: candidate pairs, memory and runtime, and check both return the same frequent pairs.

(pcy_item_counts, pcy_frequent_items, pcy_item_map, bitmap, pcy1_stats), pcy_pass1_s, pcy_pass1_MB = timed(
    pcy_pass1, flat_items, offsets, n_products, support)
(pcy_pair_keys, pcy_pair_counts, pcy2_stats), pcy_pass2_s, pcy_pass2_MB = timed(
    pcy_pass2, flat_items, offsets, pcy_frequent_items, pcy_item_map, bitmap, support, n_products)
pcy_candidates = count_pcy_candidates(pcy_frequent_items, bitmap)

same_pairs = np.array_equal(pair_keys, pcy_pair_keys) and np.array_equal(pair_counts, pcy_pair_counts)

pcy_comparison = pd.DataFrame([
    {"method": "Apriori", "candidate_pairs": apriori_stats["candidate_pairs"],
     "counting_structure_MB": apriori_stats["structure_MB"],
     "pass1_s": apriori_pass1_s, "pass2_s": apriori_pass2_s, "total_s": apriori_pass1_s + apriori_pass2_s,
     "pass1_peak_MB": apriori_pass1_MB, "pass2_peak_MB": apriori_pass2_MB, "frequent_pairs": len(pair_keys)},
    {"method": "PCY", "candidate_pairs": pcy_candidates,
     "counting_structure_MB": pcy1_stats["bitmap_MB"] + 12 * pcy_candidates / 1e6,
     "pass1_s": pcy_pass1_s, "pass2_s": pcy_pass2_s, "total_s": pcy_pass1_s + pcy_pass2_s,
     "pass1_peak_MB": pcy_pass1_MB, "pass2_peak_MB": pcy_pass2_MB, "frequent_pairs": len(pcy_pair_keys)}])

print ("~~~~ Apriori vs. PCY (support", support, "baskets) ~~~~")
print ("Frequent buckets:\t\t\t", round(pcy1_stats["frequent_bucket_share"] * 100, 2), "% of", N_BUCKETS)
print ("Bucket counts in pass 1:\t\t", round(pcy1_stats["bucket_array_MB"], 1), "MB, bitmap after pass 1:", round(pcy1_stats["bitmap_MB"], 1), "MB")
print ("Candidate pairs removed by PCY:\t\t", round(100 * (1 - pcy_candidates / apriori_stats["candidate_pairs"]), 1), "%")
print ("Pairs actually counted in PCY pass 2:\t", pcy2_stats["counted_pairs"])
print ("Both methods return the same frequent pairs and counts:", same_pairs)
display (pcy_comparison.round(3))

os.makedirs(results_path, exist_ok = True)
pcy_comparison.to_csv(os.path.join(results_path, "apriori_vs_pcy.csv"), index = False)

How to read this comparison: PCY's pass 1 does more work than Apriori's, because it generates and hashes every pair in every basket (not just pairs of frequent items) and keeps a bucket array in memory. Its pass 2 can only make up for this if the bitmap removes a large share of the candidate pairs.

- If the frequent buckets are a small share of all buckets, PCY removes most candidate pairs, and its counting structure is smaller than Apriori's triangular array (it must remove at least 2/3 of the candidates to break even, since each candidate takes 12 bytes instead of 4).
- Compare the total runtime of both passes. If PCY's pass 2 saving is smaller than its extra pass 1 cost, PCY is slower overall on this data, even if it uses less memory for counting.

At our support threshold the number of frequent items m is small, so Apriori's triangular array already fits easily in memory. PCY's advantage grows at lower thresholds, where m and the triangular array (which grows with m^2) become much larger.

## Association rules

From the frequent itemsets we build rules I -> j, where I is one or two items and j is a single item. From section 6.1.3:

- Confidence of I -> j: support(I ∪ {j}) / support(I), the fraction of baskets with I that also contain j.
- Interest of I -> j: confidence - support(j) / n, how much more likely j is given I than in a random basket.
- Lift of I -> j: confidence / (support(j) / n), the same comparison as a ratio. A lift of 1 means I and j are unrelated.

Sorting by confidence alone would be dominated by very popular items: since about 1 in 7 baskets contains bananas, "anything -> bananas" has a high confidence even when the two items are unrelated. So we filter on lift (and a small minimum confidence), which keeps only rules where I really makes j more likely.

Since every subset of a frequent itemset is frequent, the counts needed for every rule's antecedent are already known.

In [ ]:
# Build association rules with confidence, interest and lift, and filter them on lift.

MIN_CONFIDENCE = 0.05
MIN_LIFT = 1.5

def association_rules(itemsets, n_baskets, n_items, level):
  # Rules I -> j from frequent pairs and triples, with confidence, interest and lift (section 6.1.3). No filtering here.
  single_counts = np.zeros(n_items, dtype = np.int64)
  single_counts[itemsets[1][0]] = itemsets[1][1]
  pair_keys, pair_counts = itemsets[2]
  groups = []      # (antecedent IDs, consequent IDs, itemset counts, antecedent counts)

  # From each pair {a, b}: a -> b and b -> a.
  pairs = decode_keys(pair_keys, n_items, 2)
  for antecedent, consequent in [([0], 1), ([1], 0)]:
    groups.append((pairs[:, antecedent], pairs[:, consequent], pair_counts, single_counts[pairs[:, antecedent[0]]]))

  # From each triple {a, b, c}: {a, b} -> c, {a, c} -> b and {b, c} -> a. The antecedent pairs are frequent, so their counts are known.
  if 3 in itemsets and len(itemsets[3][0]):
    triples = decode_keys(itemsets[3][0], n_items, 3)
    for antecedent, consequent in [([0, 1], 2), ([0, 2], 1), ([1, 2], 0)]:
      antecedent_keys = triples[:, antecedent[0]] * n_items + triples[:, antecedent[1]]
      position = np.searchsorted(pair_keys, antecedent_keys)
      assert np.array_equal(pair_keys[position], antecedent_keys), "An antecedent pair is missing from the frequent pairs."
      groups.append((triples[:, antecedent], triples[:, consequent], itemsets[3][1], pair_counts[position]))

  tables = []
  for antecedent, consequent, itemset_counts, antecedent_counts in groups:
    confidence = itemset_counts / antecedent_counts
    consequent_support = single_counts[consequent] / n_baskets
    tables.append(pd.DataFrame({
        "level": level,
        "antecedent_ids": [row.tolist() for row in antecedent],
        "consequent_id": consequent,
        "itemset_size": antecedent.shape[1] + 1,
        "support_count": itemset_counts,
        "support": itemset_counts / n_baskets,
        "confidence": confidence,
        "interest": confidence - consequent_support,
        "lift": confidence / consequent_support}))
  return pd.concat(tables, ignore_index = True)

def add_names(rules, names):
  # Readable names for the antecedent and consequent of each rule.
  rules = rules.copy()
  rules["antecedent_names"] = [[names[i] for i in ids] for ids in rules["antecedent_ids"]]
  rules["consequent_name"] = names[rules["consequent_id"].to_numpy()]
  return rules

def filter_rules(rules, min_confidence = MIN_CONFIDENCE, min_lift = MIN_LIFT):
  # Keep rules where the antecedent really makes the consequent more likely, sorted by lift.
  keep = (rules["confidence"] >= min_confidence) & (rules["lift"] >= min_lift)
  return rules[keep].sort_values("lift", ascending = False).reset_index(drop = True)

product_itemsets = {1: (frequent_items.astype(np.int64), item_counts[frequent_items].astype(np.int64)),
                    2: (pair_keys, pair_counts), 3: (triple_keys, triple_counts)}
all_product_rules = association_rules(product_itemsets, n_baskets, n_products, "product")
product_rules = add_names(filter_rules(all_product_rules), product_names)

shown_columns = ["antecedent_names", "consequent_name", "support", "confidence", "lift"]
top_by_confidence = add_names(all_product_rules.nlargest(1000, "confidence"), product_names)

print ("~~~~ Product-level association rules ~~~~")
print ("Rules before filtering:\t", len(all_product_rules))
print ("Rules after filtering (confidence >=", MIN_CONFIDENCE, ", lift >=", MIN_LIFT, "):\t", len(product_rules))

print ("\nWithout a lift filter, the 1000 rules with the highest confidence mostly predict a few popular items:")
display (top_by_confidence["consequent_name"].value_counts().head(5))

print ("Top rules after filtering on lift:")
display (product_rules[shown_columns].head(15))

## Aisle and department level

The same algorithms also run at the aisle level (134 aisles) and department level (21 departments), by replacing every product in a basket with its aisle or department and removing repeats within each basket. With so few items, this is cheap, and the recommendation step needs product, aisle and department rules.

Since there are few aisles and departments, almost every pair of them appears together in at least 0.1% of baskets, so we use a higher support threshold here (1%). The same confidence and lift filters as for products are used.

In [ ]:
# Run Apriori and association rules at aisle and department level.

LEVEL_SUPPORT_FRACTION = {"aisle": 0.01, "department": 0.01}

def level_baskets(flat_items, offsets, item_map, chunk_baskets = CHUNK_BASKETS):
  # Re-express every basket with item_map codes (e.g. product -> aisle), removing repeats within each basket.
  level_items, level_sizes = [], []
  n_baskets = len(offsets) - 1
  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, _, sizes = basket_chunk(flat_items, offsets, item_map, start, stop)
    level_items.append(items)
    level_sizes.append(sizes)
  return np.concatenate(level_items), np.r_[0, np.cumsum(np.concatenate(level_sizes))].astype(np.int64)

level_rules = {}
for level in ["aisle", "department"]:
  # Integer code for each aisle or department name, and a product ID -> code map.
  level_names = np.sort(products_lookup[level].unique()).astype(object)
  product_to_level = np.full(n_products, -1, dtype = np.int32)
  product_to_level[products_lookup["product_id"].to_numpy()] = np.searchsorted(level_names, products_lookup[level].to_numpy())

  level_items, level_offsets = level_baskets(flat_items, offsets, product_to_level)
  level_support = support_threshold(LEVEL_SUPPORT_FRACTION[level], n_baskets)
  itemsets, _ = run_apriori(level_items, level_offsets, len(level_names), level_support)

  all_rules = association_rules(itemsets, n_baskets, len(level_names), level)
  level_rules[level] = add_names(filter_rules(all_rules), level_names)

  print ("~~~~", level.capitalize(), "level (support", level_support, "baskets) ~~~~")
  print ("Frequent itemsets of size 1, 2, 3:\t", [len(itemsets[k][0]) for k in (1, 2, 3)])
  print ("Rules before / after filtering:\t\t", len(all_rules), "/", len(level_rules[level]))
  display (level_rules[level][shown_columns].head(10))
  del level_items, level_offsets

All rules from the three levels are saved together in rules.parquet. The "level" column says whether a rule is about products, aisles or departments, and the IDs are product IDs, or the aisle and department codes used here (the matching names are saved next to them).

In [ ]:
# Save the filtered rules of all three levels for the recommendation step.

rules = pd.concat([product_rules, level_rules["aisle"], level_rules["department"]], ignore_index = True)
rules.to_parquet(os.path.join(save_path, "rules.parquet"))

print ("Saved", len(rules), "rules to", os.path.join(save_path, "rules.parquet"))
display (rules.groupby("level").size())

## Experiment 1: Different support thresholds

The support threshold controls how many items are frequent (m), which decides the size of the triangular array (m(m-1)/2 counts), the number of frequent pairs and triples, and the runtime. We run Apriori up to triples at several thresholds around our starting point of 0.1%.

In [ ]:
# Experiment 1: Apriori up to triples at several support thresholds.

SUPPORT_FRACTIONS = [0.005, 0.002, 0.001, 0.0005]

results = []
for fraction in SUPPORT_FRACTIONS:
  threshold = support_threshold(fraction, n_baskets)
  (itemsets, stats), elapsed, peak = timed(run_apriori, flat_items, offsets, n_products, threshold)
  results.append({"support_fraction": fraction, "support_count": threshold,
                  "frequent_items": stats["frequent_items"], "candidate_pairs": stats["candidate_pairs"],
                  "triangular_array_MB": stats["structure_MB"], "frequent_pairs": len(itemsets[2][0]),
                  "candidate_triples": stats["candidate_triples"], "frequent_triples": len(itemsets[3][0]),
                  "time_s": elapsed, "peak_MB": peak})
  print ("Finished support", fraction, "in", round(elapsed, 1), "seconds")

threshold_results = pd.DataFrame(results)
display (threshold_results.round(4))

os.makedirs(results_path, exist_ok = True)
threshold_results.to_csv(os.path.join(results_path, "apriori_thresholds.csv"), index = False)

## Experiment 2: Runtime and peak memory as the number of baskets grows

We run Apriori (up to triples) and PCY (both passes) on 10%, 25%, 50% and 100% of the baskets, keeping the support threshold at the same fraction of baskets, and record the runtime and peak memory of each.

In [ ]:
# Experiment 2: runtime and peak memory of Apriori and PCY on a growing share of baskets.

BASKET_FRACTIONS = [0.1, 0.25, 0.5, 1.0]
rng_scale = np.random.default_rng(SEED + 1)

def run_pcy(flat_items, offsets, n_items, support):
  # Both passes of PCY, returning the frequent pairs.
  _, frequent_items, item_map, bitmap, _ = pcy_pass1(flat_items, offsets, n_items, support)
  return pcy_pass2(flat_items, offsets, frequent_items, item_map, bitmap, support, n_items)

results = []
for fraction in BASKET_FRACTIONS:
  rows = np.sort(rng_scale.choice(n_baskets, size = int(fraction * n_baskets), replace = False))
  scale_items, scale_offsets = subset_baskets(flat_items, offsets, rows)
  threshold = support_threshold(SUPPORT_FRACTION, len(rows))

  _, apriori_s, apriori_MB = timed(run_apriori, scale_items, scale_offsets, n_products, threshold)
  _, pcy_s, pcy_MB = timed(run_pcy, scale_items, scale_offsets, n_products, threshold)
  results.append({"basket_fraction": fraction, "baskets": len(rows), "support_count": threshold,
                  "apriori_s": apriori_s, "apriori_peak_MB": apriori_MB, "pcy_s": pcy_s, "pcy_peak_MB": pcy_MB})
  print ("Finished", len(rows), "baskets: Apriori", round(apriori_s, 1), "s, PCY", round(pcy_s, 1), "s")
  del scale_items, scale_offsets

basket_scaling = pd.DataFrame(results)
display (basket_scaling.round(3))

os.makedirs(results_path, exist_ok = True)
basket_scaling.to_csv(os.path.join(results_path, "frequent_itemsets_scaling.csv"), index = False)

fig, axes = plt.subplots(1, 2, figsize = (11, 4))
for method, label in [("apriori", "Apriori"), ("pcy", "PCY")]:
  axes[0].plot(basket_scaling["baskets"], basket_scaling[method + "_s"], marker = "o", label = label)
  axes[1].plot(basket_scaling["baskets"], basket_scaling[method + "_peak_MB"], marker = "o", label = label)
axes[0].set_ylabel("Time (seconds)")
axes[0].set_title("Runtime")
axes[1].set_ylabel("Peak memory (MB)")
axes[1].set_title("Peak memory")
for ax in axes:
  ax.set_xlabel("Baskets")
  ax.legend()
  ax.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

## SON algorithm

The SON algorithm (section 6.4.4) finds the same frequent itemsets in two passes over data that is split into chunks:

1. Run Apriori on each chunk separately, with the support threshold scaled down to the chunk's size (fraction x chunk baskets). Every itemset that is frequent in at least one chunk becomes a candidate.
2. Count every candidate over the full data, and keep the ones that reach the full support threshold.

There are no false negatives: an itemset that is frequent in the full data must reach the scaled threshold in at least one chunk, since otherwise its total count would be below the full threshold. The second pass removes the false positives, so SON returns exactly the same frequent itemsets as Apriori, which we check below.

This is the natural way to run frequent-itemset mining with MapReduce, since each chunk can be processed by a different machine (see the MapReduce section at the end).

In [ ]:
# SON: Apriori on each chunk with a scaled threshold, then a second counting pass over the full data.

N_SON_CHUNKS = 4

def count_candidates(flat_items, offsets, candidate_keys, n_items, k, chunk_baskets = CHUNK_BASKETS):
  # Count how many baskets contain each candidate k-itemset (candidate_keys sorted, encoded in base n_items).
  counts = np.zeros(len(candidate_keys), dtype = np.int64)
  if len(candidate_keys) == 0:
    return counts

  # Keep only items that are part of some candidate, with their original IDs.
  candidate_items = np.unique(decode_keys(candidate_keys, n_items, k)).astype(np.int32)
  item_map = np.full(n_items, -1, dtype = np.int32)
  item_map[candidate_items] = candidate_items

  n_baskets = len(offsets) - 1
  for start in range(0, n_baskets, chunk_baskets):
    stop = min(start + chunk_baskets, n_baskets)
    items, starts, sizes = basket_chunk(flat_items, offsets, item_map, start, stop)
    for combos in basket_combinations(items, starts, sizes, k):
      keys = combos[:, 0].astype(np.int64)
      for column in range(1, k):
        keys = keys * n_items + combos[:, column]
      position = np.minimum(np.searchsorted(candidate_keys, keys), len(candidate_keys) - 1)
      found = candidate_keys[position] == keys
      counts += np.bincount(position[found], minlength = len(candidate_keys))
  return counts

def son(flat_items, offsets, n_items, support_fraction, n_chunks = N_SON_CHUNKS):
  # Returns {k: (sorted keys, counts)} of frequent itemsets up to triples, and stats about each phase.
  n_baskets = len(offsets) - 1
  bounds = np.linspace(0, n_baskets, n_chunks + 1).astype(np.int64)

  # Phase 1 (first Map): Apriori on each chunk with a scaled threshold. The union of local results is the candidate set.
  start_time = time.perf_counter()
  local_keys = {1: [], 2: [], 3: []}
  for low, high in zip(bounds[:-1], bounds[1:]):
    chunk_items = flat_items[offsets[low]:offsets[high]]
    chunk_offsets = offsets[low:high + 1] - offsets[low]
    local_itemsets, _ = run_apriori(chunk_items, chunk_offsets, n_items, support_threshold(support_fraction, high - low))
    for k in local_keys:
      local_keys[k].append(local_itemsets[k][0])
  candidates = {k: np.unique(np.concatenate(keys)) for k, keys in local_keys.items()}
  phase1_s = time.perf_counter() - start_time

  # Phase 2 (second Map and Reduce): count every candidate on the full data, and keep the truly frequent ones.
  start_time = time.perf_counter()
  support = support_threshold(support_fraction, n_baskets)
  counts = {1: np.bincount(flat_items, minlength = n_items)[candidates[1]].astype(np.int64)}
  for k in (2, 3):
    counts[k] = count_candidates(flat_items, offsets, candidates[k], n_items, k)
  itemsets = {k: (candidates[k][counts[k] >= support], counts[k][counts[k] >= support]) for k in candidates}
  phase2_s = time.perf_counter() - start_time

  stats = {"phase1_s": phase1_s, "phase2_s": phase2_s}
  for k in candidates:
    stats[f"candidates_{k}"] = len(candidates[k])
  return itemsets, stats

(son_itemsets, son_stats), son_s, son_MB = timed(son, flat_items, offsets, n_products, SUPPORT_FRACTION)
same_as_apriori = all(np.array_equal(son_itemsets[k][0], product_itemsets[k][0]) and
                      np.array_equal(son_itemsets[k][1], product_itemsets[k][1]) for k in (1, 2, 3))

son_results = pd.DataFrame([{"chunks": N_SON_CHUNKS, "support_count": support, **son_stats,
                             "frequent_items": len(son_itemsets[1][0]), "frequent_pairs": len(son_itemsets[2][0]),
                             "frequent_triples": len(son_itemsets[3][0]), "total_s": son_s, "peak_MB": son_MB}])

print ("~~~~ SON with", N_SON_CHUNKS, "chunks ~~~~")
print ("Same frequent itemsets and counts as Apriori:", same_as_apriori)
display (son_results.round(3))

os.makedirs(results_path, exist_ok = True)
son_results.to_csv(os.path.join(results_path, "son.csv"), index = False)

## Link to MapReduce

SON maps directly onto two MapReduce jobs (step 5):

- **First Map:** each mapper gets one chunk of baskets and runs Apriori on it with the scaled threshold, emitting (itemset, 1) for each locally frequent itemset.
- **First Reduce:** combine the emitted itemsets (ignoring the values) into the set of candidate itemsets.
- **Second Map:** each mapper gets the full candidate set and one chunk of baskets, and emits (candidate, count in this chunk).
- **Second Reduce:** sum the counts of each candidate and keep those that reach the full support threshold.

Apriori and PCY's first pass are also a map step (count items or hash pairs per chunk) followed by a reduce step that sums the counts.

In [ ]:
# Clear RAM once the file is finished running

import gc

del flat_items, offsets, item_counts, frequent_items, item_map, products_lookup, product_names
del pair_keys, pair_counts, triple_keys, triple_counts, product_itemsets, son_itemsets
del pcy_item_counts, pcy_frequent_items, pcy_item_map, bitmap, pcy_pair_keys, pcy_pair_counts
del all_product_rules, product_rules, level_rules, rules, top_by_confidence
del check_keys, sample_items, sample_offsets, our_itemsets, our_sets
del rng_check, rng_scale

gc.collect()